<p class="lead">A hypothesis test answers one narrow question: <em>could what I am seeing plausibly have been produced by chance alone?</em> It does not say whether the effect matters, whether the experiment was designed correctly, or whether the comparison was decided before or after looking at the data — and most of the damage done by testing in the biomedical and chemical literature comes from reading the answer to the narrow question as an answer to the broad ones. This unit builds testing from a single idea, the <strong>permutation test</strong>, from which every classical test turns out to be a computational shortcut, and then spends most of its length on the three things that decide whether a test result is worth anything: the <strong>effect size with its interval</strong>, the <strong>multiplicity</strong> of the comparisons that were actually made, and the <strong>power</strong> the design ever had. The centrepiece is an experiment on the laboratory's own simulation data in which the same comparison yields p = 1.1 × 10<sup>−26</sup> and p = 0.69 depending on whether frames or replicas are counted as the experimental unit — and only one of those two numbers is admissible.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad construye las pruebas de hipótesis a partir de una sola idea, la prueba de permutación, y muestra que las pruebas clásicas (t de Student, Welch, pareada, Mann-Whitney, Wilcoxon, ANOVA, Kruskal-Wallis, ji-cuadrado y exacta de Fisher) son atajos de cálculo de esa misma idea, cada uno con sus supuestos. Se explica qué es y qué no es un valor p, su relación con el intervalo de confianza, y se calculan tamaños de efecto con intervalos (d de Cohen, g de Hedges, delta de Cliff, correlación biserial de rangos, razón de momios) con el paquete <code>effectsize</code>. Se aborda el problema de las comparaciones múltiples sobre mil genes de una matriz de expresión con diseño pareado (Bonferroni, Holm y Benjamini-Hochberg a mano y con <code>p.adjust</code>, FWER frente a FDR, valores q), el p-hacking y el jardín de senderos que se bifurcan, la potencia estadística estimada por simulación para 3 y 10 réplicas de dinámica molecular, y la maldición del ganador. El experimento central compara la misma hipótesis evaluada por fotograma y por réplica. Esta es la edición en R; la edición en Python contiene el mismo material.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Formulate a null hypothesis, build its null distribution by permutation, and recognise the classical tests as special cases of that construction.</li>
<li>Choose between Student, Welch, paired, Mann–Whitney, Wilcoxon, ANOVA, Kruskal–Wallis, chi-square and Fisher tests, and check the assumptions each one makes.</li>
<li>Interpret a p-value correctly, state its relationship to the confidence interval, and explain why "significant" is not "important".</li>
<li>Compute Cohen's d, Hedges' g, Cliff's delta, the rank-biserial correlation and an odds ratio, each with a confidence interval.</li>
<li>Correct for multiplicity with Bonferroni, Holm and Benjamini–Hochberg, distinguish family-wise error from false discovery rate, and compute q-values.</li>
<li>Recognise p-hacking, the garden of forking paths and HARKing, and write a pre-registered analysis plan for a thesis.</li>
<li>Estimate statistical power by simulation, decide how many replicas or compounds an experiment needs, and anticipate the winner's curse.</li>
<li>Identify the experimental unit of a study and avoid pseudoreplication.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read linearly, running every chunk; then re-run the permutation chunks with a different seed and watch which conclusions move. Everything runs on base R's <code>stats</code><sup class="c"><a href="#r1">[1]</a></sup> plus <code>data.table</code>, <code>ggplot2</code>, <code>Matrix</code> (for the fingerprints), <code>car</code> (Levene's test) and <code>effectsize</code><sup class="c"><a href="#r35">[35]</a></sup>; in a fresh Google Colab R runtime run once: <code>install.packages(c("data.table","ggplot2","Matrix","car","effectsize","R.utils"))</code>. This unit assumes <a href="../U7_Estimation/index.html">Unit 7</a> (sampling distributions, bootstrap, autocorrelation) and feeds <a href="../U9_Linear_Models/index.html">Unit 9</a>, where the same tests reappear as questions about regression coefficients.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(Matrix); library(car); library(effectsize)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_bw(base_size = 9) +
          theme(panel.grid.minor = element_blank(),
                plot.title = element_text(face = "bold", size = 10)))
pfmt <- function(p) ifelse(p < 1e-300, "< 1e-300", sprintf("%.3g", p))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| Matrix", as.character(packageVersion("Matrix")),
    "| car", as.character(packageVersion("car")),
    "| effectsize", as.character(packageVersion("effectsize")), "\n")



The helper `pfmt` exists because R's `t.test` returns a p-value of exactly 0 when the true value falls below the smallest representable double (about 10<sup>−308</sup>); printing "0" in a paper is wrong, and this unit will meet that case twice.



In [ ]:
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
D    <- function(f) data_file(f)
rock <- fread(D("rock1_chembl_curated.csv")); exact <- rock[censored == FALSE]
md_obs <- fread(D("md_trpcage_observables.csv"))
expr <- fread(D("geo_periodontitis_expr.csv"))
meta <- fread(D("geo_periodontitis_meta.csv"))
lib  <- fread(D("screening_library.csv"))
cat("ROCK1 compounds with exact potency :", nrow(exact), "\n")
cat("MD frames                          :", nrow(md_obs), "=", uniqueN(md_obs$condition),
    "conditions x", uniqueN(md_obs$replica), "replicas x",
    md_obs[, .N, by = .(condition, replica)]$N[1], "frames\n")
cat("expression matrix                  :", nrow(expr), "genes x", ncol(expr) - 1, "samples\n")
cat("samples                            :", paste(names(table(meta$status)), table(meta$status),
                                                   collapse = ", "),
    "| patients:", uniqueN(meta$patient_id), "\n")
cat("screening library                  :", nrow(lib), "molecules,", sum(lib$label),
    "actives,", sum(lib$label == 0), "decoys\n")



Four tables, three kinds of comparison: 2,764 compounds with a measured potency, 3,000 simulation frames in six runs, 5,000 genes measured in 138 samples from 69 patients, and a 12,018-molecule screening library with 1,327 known actives.

## 1. What a test is, and what the laboratory tests

A statistical test has four parts, and writing all four down before computing anything removes most of the confusion:

1. a **null hypothesis** \( H_0 \) — a complete, simulatable description of "nothing is going on" (the labels carry no information; the two conditions have the same distribution);
2. a **test statistic** \( T \) — any number computed from the data that is large when the effect is present (a difference of means, a difference of medians, an area under a ROC curve);
3. the **null distribution** — the distribution of \( T \) *if \( H_0 \) were true*, obtained either from a formula or by simulation;
4. the **p-value** — the probability, under \( H_0 \), of a test statistic at least as extreme as the one observed.

<div class="box bridge"><h4>Domain context — the three comparisons this laboratory makes</h4>
<p><b>(a) Actives versus decoys.</b> In <strong>virtual screening</strong> a computational score ranks a library of molecules; to know whether the score works, it is run on a set of known <strong>actives</strong> (molecules measured to bind the target) mixed with <strong>decoys</strong> (molecules assumed inactive, usually chosen to resemble the actives physically but to be structurally different). Our frozen library holds 1,327 ROCK1 actives and 10,691 decoys. The score used below is the <strong>maximum Tanimoto similarity</strong> of each molecule to a small set of 25 reference actives: the Tanimoto coefficient compares two molecular fingerprints — bit vectors recording which substructures a molecule contains — and ranges from 0 (nothing in common) to 1 (identical fingerprints).</p>
<p><b>(b) Two simulation conditions.</b> Three replica molecular-dynamics runs at 300 K against three at 340 K, as described in <a href="../U7_Estimation/index.html">Unit 7</a>: the question is whether the protein is measurably less stable at the higher temperature after 1 ns. A <strong>replica</strong> is an independent repetition of the same simulation with a different random seed.</p>
<p><b>(c) Diseased versus healthy tissue.</b> 69 patients each contributed one <strong>gingival</strong> (gum) tissue sample from a site affected by <strong>periodontitis</strong> — a chronic inflammatory disease of the tissues around the teeth — and one from a clinically healthy site<sup class="c"><a href="#r2">[2]</a></sup>. Each sample's RNA was measured on a microarray, giving a log<sub>2</sub> expression value for 5,000 genes. Because both samples come from the same patient, the design is <strong>paired</strong>, and ignoring that pairing throws away most of the information.</p></div>

## 2. The permutation test is the whole idea

If the labels are meaningless, then any re-assignment of the labels to the same data is as likely as the one we observed. So: compute the statistic on the real labelling; shuffle the labels many times and recompute; the p-value is the fraction of shuffles that reach or exceed the real value. That is the entire procedure<sup class="c"><a href="#r3">[3]</a></sup>. It makes no assumption about normality, requires no formula, and works for any statistic you can compute — a difference of medians, an enrichment factor, an AUC.

<div class="box warn"><h4>Substitution in the R edition — fingerprints</h4>
<p>The Python edition computes Morgan fingerprints with RDKit inside the notebook. R has no RDKit binding in this environment, so the fingerprints (radius 2, 2,048 bits) were generated once with RDKit and frozen as a sparse <code>(id, bit)</code> file, <code>data/screening_morgan_sparse.csv.gz</code>, whose provenance is recorded in <a href="../../data/manifest_fingerprints.json">data/manifest_fingerprints.json</a>; <code>data/rock1_curated.sdf.gz</code> ships the structures so that a reader with ChemmineR or rcdk can recompute them. Rebuilding the bit matrix with <code>Matrix::sparseMatrix()</code> and computing Tanimoto similarities from it is three lines of sparse algebra, written out in the next chunk. Nothing else in the unit depends on chemistry software.</p></div>



In [ ]:
fp  <- fread(D("screening_morgan_sparse.csv.gz"))                   ## columns: id, bit (0-based)
M   <- sparseMatrix(i = match(fp$id, lib$inchikey), j = fp$bit + 1L, x = 1,
                    dims = c(nrow(lib), 2048))                      ## molecules x bits
pc  <- rowSums(M)                                                   ## popcount per molecule
set.seed(0)
ref <- sample(which(lib$label == 1), 25)                            ## 25 reference actives
inter <- as.matrix(M %*% t(M[ref, , drop = FALSE]))                 ## |A and B| for every pair
tani  <- inter / (outer(pc, pc[ref], "+") - inter)                   ## |A and B| / |A or B|
score <- apply(tani, 1, max)
keep  <- rep(TRUE, nrow(lib)); keep[ref] <- FALSE                    ## never score a reference on itself
y <- lib$label[keep]; s <- score[keep]
act <- s[y == 1]; dec <- s[y == 0]
cat(sprintf("fingerprint matrix: %d molecules x %d bits, %d set bits (mean %.1f per molecule)\n",
            nrow(M), ncol(M), nrow(fp), mean(pc)))
cat(sprintf("actives: n = %d, mean score %.3f, median %.3f\n", length(act), mean(act), median(act)))
cat(sprintf("decoys : n = %d, mean score %.3f, median %.3f\n", length(dec), mean(dec), median(dec)))
cat(sprintf("observed difference of means: %.4f\n", mean(act) - mean(dec)))



Each molecule carries on average 51.4 of the 2,048 bits, so the matrix is 2.5 % dense and the whole similarity calculation is one sparse matrix product. The 1,302 actives (25 of the 1,327 are references and were removed) score 0.438 on average against 0.179 for the 10,691 decoys, a raw difference of 0.2594 similarity units. Is that difference bigger than label noise?



In [ ]:
permutation_test <- function(a, b, stat = function(x, y) mean(x) - mean(y),
                             n_perm = 5000, seed = 0, alternative = "greater") {
  set.seed(seed)
  pooled <- c(a, b); n_a <- length(a); obs <- stat(a, b)
  null <- replicate(n_perm, { p <- sample(pooled); stat(p[1:n_a], p[(n_a + 1):length(p)]) })
  p_val <- if (alternative == "greater") (1 + sum(null >= obs)) / (n_perm + 1)
           else (1 + sum(abs(null) >= abs(obs))) / (n_perm + 1)      ## the +1 keeps the p-value valid
  list(obs = obs, p = p_val, null = null)
}
r1 <- permutation_test(act, dec, n_perm = 5000, seed = 0)
p <- ggplot(data.frame(null = r1$null), aes(null)) +
  geom_histogram(bins = 50, fill = "grey75", colour = NA) +
  geom_vline(xintercept = 0, linetype = 3) +
  coord_cartesian(xlim = c(-0.02, 0.28)) +
  annotate("segment", x = r1$obs, xend = r1$obs, y = 0, yend = Inf,
           colour = PAL["orange"], linewidth = 1) +
  annotate("text", x = r1$obs, y = Inf, vjust = 1.6, hjust = 1.05, size = 2.8,
           label = sprintf("observed = %.3f", r1$obs)) +
  labs(x = "difference of mean similarity (actives - decoys) under shuffled labels", y = "count",
       title = "The observed effect is nowhere near the null distribution")
print(p)
cat(sprintf("null distribution: mean %.5f, SD %.5f, range %.4f to %.4f\n",
            mean(r1$null), sd(r1$null), min(r1$null), max(r1$null)))
cat(sprintf("observed statistic %.4f -> %.1f null standard deviations away\n",
            r1$obs, (r1$obs - mean(r1$null)) / sd(r1$null)))
cat(sprintf("permutation p-value (one-sided) = %.5f  <- the smallest 5,000 permutations can report is %.5f\n",
            r1$p, 1 / 5001))



<p class="caption"><b>Figure 1.</b> Null distribution of the difference in mean similarity score between actives and decoys, built by shuffling the active/decoy labels 5,000 times (the x-axis is cut at 0.28 so that both the null cloud and the observed value are visible). Under the null the difference is centred on zero with a standard deviation of 0.0031 and never exceeds 0.012; the observed difference of 0.2594 sits 83.5 null standard deviations to the right. The reported p-value is 1/5001 = 0.0002 — the smallest value this many permutations can produce — and the honest way to write it in a paper is <em>p &lt; 2 × 10<sup>−4</sup> (5,000 permutations)</em>, not "p = 0".</p>

Two details of the implementation matter. The **+1 in the numerator and denominator** is not cosmetic: without it a p-value of exactly zero is possible, which is a probability statement that cannot be true, and the corrected version is the valid Monte-Carlo p-value. And the **resolution** of a permutation test is bounded by the number of permutations: 5,000 shuffles can never report anything below 2 × 10<sup>−4</sup>, so if you need a smaller p-value you must either run more permutations or use a parametric approximation.

### 2.1 A worked example traced by hand

The permutation idea is clearest when the number of permutations is small enough to enumerate completely. Take the six replica means of the C-alpha RMSD from the molecular-dynamics dataset — 300 K: 0.1098, 0.1595, 0.2515 nm; 340 K: 0.1211, 0.1884, 0.1504 nm — and ask whether temperature changed the mean.

Observed difference of means: (0.1098 + 0.1595 + 0.2515)/3 − (0.1211 + 0.1884 + 0.1504)/3 = 0.17360 − 0.15330 = **0.02030 nm**.

There are \( \binom{6}{3} = 20 \) ways of splitting six numbers into two groups of three. Under the null hypothesis that the temperature label is irrelevant, each of those 20 labellings was equally likely. Here are the first few, sorted by the absolute difference they produce:

| rank | group "A" | mean A | mean B | \|difference\| |
|---|---|---|---|---|
| 1 | 0.2515, 0.1884, 0.1595 | 0.1998 | 0.1271 | 0.0727 |
| 2 | 0.2515, 0.1884, 0.1504 | 0.1968 | 0.1301 | 0.0666 |
| 3 | 0.2515, 0.1884, 0.1211 | 0.1870 | 0.1399 | 0.0471 |
| … | … | … | … | … |
| — | **0.1098, 0.1595, 0.2515** (the observed one) | **0.1736** | **0.1533** | **0.0203** |

The two-sided p-value is the proportion of the 20 labellings whose absolute difference is at least 0.0203. The next chunk enumerates all twenty.



In [ ]:
per_run <- md_obs[, .(mean_rmsd = mean(rmsd_ca_nm)), by = .(condition, replica)]
A <- per_run[condition == "A", mean_rmsd]          ## 300 K replica means (nm)
B <- per_run[condition == "B", mean_rmsd]          ## 340 K replica means (nm)
allv <- c(A, B); obs_d <- mean(A) - mean(B)
cmb  <- combn(6, 3)
enum <- data.table(
  group_A  = apply(cmb, 2, function(k) paste(sprintf("%.4f", allv[k]), collapse = ", ")),
  meanA    = apply(cmb, 2, function(k) mean(allv[k])),
  meanB    = apply(cmb, 2, function(k) mean(allv[-k])))
enum[, abs_diff := abs(meanA - meanB)]
setorder(enum, -abs_diff)
p_exact <- mean(enum$abs_diff >= abs(obs_d) - 1e-12)
cat(sprintf("observed difference %.5f nm\n", obs_d))
cat(sprintf("number of possible labellings: %d\n", nrow(enum)))
cat(sprintf("labellings at least as extreme: %d  ->  exact two-sided p = %.3f\n",
            sum(enum$abs_diff >= abs(obs_d) - 1e-12), p_exact))
cat(sprintf("smallest two-sided p attainable with 3 vs 3 replicas: 2/20 = %.2f\n", 2 / 20))
print(head(enum[, .(group_A, meanA, meanB, abs_diff)], 4))
cat("rank of the observed labelling:",
    which(abs(enum$abs_diff - abs(obs_d)) < 1e-12)[1], "of 20\n")



The hand calculation and the code agree: the top of the enumeration is 0.0727, 0.0727, 0.0666, 0.0666 — every split and its mirror image give the same absolute difference, so the values come in pairs — the observed split ranks 15th of 20, and the exact two-sided p-value is 0.800. More importantly, the printed line about resolution is a fact about the **design**, not about the data: with three replicas per condition there are only 20 labellings, so the smallest two-sided p-value the experiment can possibly produce is 2/20 = 0.10. **A 3-versus-3 permutation test can never be significant at the 0.05 level, whatever the data look like.** No amount of computation fixes that; only more replicas do, which is the subject of §8.

## 3. The classical tests are special cases

Before computers, enumerating permutations was impossible, so the field derived formulas that approximate the permutation null under extra assumptions. Each classical test is a shortcut of this kind, and knowing which assumption buys which shortcut is how you choose one.

| test | compares | R function | key assumptions |
|---|---|---|---|
| Student's *t* | two means | `t.test(var.equal = TRUE)` | normal data, **equal variances**, independence |
| Welch's *t* | two means | `t.test()` (the default) | normal data, unequal variances allowed<sup class="c"><a href="#r4">[4]</a></sup> |
| paired *t* | mean of within-pair differences | `t.test(paired = TRUE)` | normal *differences*, independent pairs |
| Mann–Whitney *U* | stochastic ordering | `wilcox.test(x, y)` | independent, continuous-ish data<sup class="c"><a href="#r5">[5]</a></sup> |
| Wilcoxon signed-rank | within-pair differences | `wilcox.test(d)` | symmetric differences<sup class="c"><a href="#r6">[6]</a></sup> |
| one-way ANOVA | *k* means | `aov()` / `anova(lm())` | normal, equal variances, independence |
| Kruskal–Wallis | *k* distributions by rank | `kruskal.test()` | independence<sup class="c"><a href="#r7">[7]</a></sup> |
| chi-square | counts in a contingency table | `chisq.test()` | expected counts ≳ 5<sup class="c"><a href="#r8">[8]</a></sup> |
| Fisher's exact | counts in a 2 × 2 table | `fisher.test()` | fixed margins |
| permutation | anything | written by hand | exchangeability under the null |

The first comparison checks the claim directly: on the actives-versus-decoys data, do the shortcut and the permutation test agree?



In [ ]:
r1m <- permutation_test(act, dec, stat = function(x, y) median(x) - median(y), n_perm = 5000, seed = 1)
t_eq <- t.test(act, dec, var.equal = TRUE, alternative = "greater")
t_we <- t.test(act, dec, alternative = "greater")
u    <- suppressWarnings(wilcox.test(act, dec, alternative = "greater"))
comp <- data.table(
  test = c("Student t (equal var)", "Welch t", "Mann-Whitney U",
           "permutation (means, 5000)", "permutation (medians, 5000)"),
  statistic = c(t_eq$statistic, t_we$statistic, u$statistic, r1$obs, r1m$obs),
  p = c(pfmt(t_eq$p.value), pfmt(t_we$p.value), pfmt(u$p.value),
        sprintf("%.5f", r1$p), sprintf("%.5f", r1m$p)))
grp <- factor(rep(c("active", "decoy"), c(length(act), length(dec))))
lev <- leveneTest(c(act, dec) ~ grp)
cat(sprintf("variance ratio actives/decoys: %.2f  (Levene F = %.0f, p = %s)\n",
            var(act) / var(dec), lev[1, 2], pfmt(lev[1, 3])))
cat(sprintf("group sizes: %d vs %d | Welch df = %.0f\n", length(act), length(dec), t_we$parameter))
knitr::kable(comp, digits = 4)



All five routes agree that the score separates actives from decoys. The agreement is the point: when the assumptions hold well enough, the parametric shortcut reproduces the permutation answer at a fraction of the cost. Two R-specific observations. First, the two t-statistics differ enormously — 137.0 for Student against 55.1 for Welch — because the variance ratio between the groups is 24.4 and Levene's test rejects equality decisively; with 1,302 actives against 10,691 decoys the pooled variance of the Student version is dominated by the tight decoy group, which inflates the statistic. **Welch's test is the default in R for exactly this reason**<sup class="c"><a href="#r4">[4]</a></sup>, and you have to ask for the wrong test explicitly. Second, both parametric p-values underflow to zero in double precision, which is why they print as "< 1e-300" rather than as a number.

What none of the five numbers tells us is whether the score is *useful*. They all answer "is the difference exactly zero?", a question whose answer was known in advance. Section 5 replaces them with a number that answers the real question.

### 3.1 Paired data: the single most valuable design feature

When each measurement in one group has a natural partner in the other — the same patient, the same compound in two assays, the same system before and after a mutation — the **paired** test removes between-subject variation by analysing the within-pair differences. The gain is usually larger than anything else available to the experimenter.



In [ ]:
gmeta <- meta[, .(sample_id, patient_id, status)]
dis <- gmeta[status == "diseased"][order(patient_id)]
hea <- gmeta[status == "healthy"][order(patient_id)]
stopifnot(identical(dis$patient_id, hea$patient_id))      ## same patient order in both vectors
gene  <- "PLA2R1"
rowg  <- expr[symbol == gene]
d_vals <- as.numeric(rowg[, dis$sample_id, with = FALSE])
h_vals <- as.numeric(rowg[, hea$sample_id, with = FALSE])
dvec   <- d_vals - h_vals
op <- par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1), cex = 0.75)
plot(NA, xlim = c(0.8, 2.2), ylim = range(c(d_vals, h_vals)), xaxt = "n",
     xlab = "", ylab = sprintf("log2 expression of %s", gene), main = "each line is one patient")
axis(1, at = c(1, 2), labels = c("healthy site", "diseased site"))
segments(1, h_vals, 2, d_vals, col = "#99999999")
points(rep(1, length(h_vals)), h_vals, pch = 19, cex = 0.5, col = PAL["green"])
points(rep(2, length(d_vals)), d_vals, pch = 19, cex = 0.5, col = PAL["orange"])
hist(dvec, breaks = 22, col = PAL["blue"], border = NA, main = "the paired test works on this histogram",
     xlab = "within-patient difference (diseased - healthy)", ylab = "patients")
abline(v = 0, lty = 2, lwd = 1.5)
par(op)
set.seed(11)
flips <- replicate(5000, mean(sample(c(-1, 1), length(dvec), replace = TRUE) * dvec))
cat(sprintf("unpaired Welch t : t = %6.3f, p = %s\n", t.test(d_vals, h_vals)$statistic,
            pfmt(t.test(d_vals, h_vals)$p.value)))
cat(sprintf("paired t         : t = %6.3f, p = %s\n", t.test(d_vals, h_vals, paired = TRUE)$statistic,
            pfmt(t.test(d_vals, h_vals, paired = TRUE)$p.value)))
cat(sprintf("Wilcoxon signed  : V = %6.0f, p = %s\n", wilcox.test(dvec)$statistic,
            pfmt(wilcox.test(dvec)$p.value)))
cat(sprintf("sign-flip permutation p = %.4f\n", (1 + sum(abs(flips) >= abs(mean(dvec)))) / 5001))
cat(sprintf("mean within-patient difference %.3f log2 units\n", mean(dvec)))
cat(sprintf("between-patient SD %.3f | within-pair SD %.3f -> pairing removes %.0f%% of the noise\n",
            sd(c(d_vals, h_vals)), sd(dvec), 100 * (1 - sd(dvec) / sd(c(d_vals, h_vals)))))



<p class="caption"><b>Figure 2.</b> Expression of <i>PLA2R1</i> in 69 matched pairs of gingival tissue. Left: each grey line joins the healthy and diseased sample of one patient; the lines are mostly parallel and mostly descending, which is exactly the situation the paired test exploits — patients differ widely in overall level, but within a patient the direction is consistent. Right: the histogram of the 69 within-patient differences is shifted clearly below zero, with a mean of −0.980 log<sub>2</sub> units (a two-fold reduction). The printed statistics show the consequence: on identical numbers the paired p-value is 1.3 × 10<sup>−20</sup> against 6.1 × 10<sup>−17</sup> unpaired, because working within patients removes 21 % of the standard deviation — and that modest-sounding reduction is amplified by the √n in the test statistic.</p>

The correct permutation null for a paired design is not a shuffle of the group labels but a **random flip of the sign of each within-pair difference**, because under the null the diseased and healthy label within a patient are exchangeable. The sign-flip p-value hits its resolution floor of 1/5001 = 0.0002, consistent with the Wilcoxon and paired-t results. Choosing the wrong exchangeability structure is the most common way to get a permutation test wrong.

### 3.2 More than two groups, and counts



In [ ]:
cnt  <- sort(table(exact$scaffold_smiles), decreasing = TRUE)
top5 <- names(cnt)[1:5]
ser  <- rbindlist(lapply(seq_along(top5), function(i)
  data.table(series = sprintf("S%d", i), pActivity = exact[scaffold_smiles == top5[i], pActivity])))
ser[, series := factor(series, levels = sprintf("S%d", 1:5))]
set.seed(12)
p <- ggplot(ser, aes(series, pActivity)) +
  geom_boxplot(outlier.shape = NA, fill = "grey92") +
  geom_jitter(width = 0.16, size = 0.5, alpha = 0.4, colour = PAL["blue"]) +
  stat_summary(fun = mean, geom = "point", shape = 23, size = 1.8, fill = PAL["orange"]) +
  labs(x = NULL, y = "pActivity",
       title = "Five Bemis-Murcko scaffold series of the ROCK1 dataset")
print(p)
fit_aov <- summary(aov(pActivity ~ series, ser))[[1]]
kw <- kruskal.test(pActivity ~ series, ser)
cat(sprintf("one-way ANOVA   : F = %.2f, p = %s\n", fit_aov[1, 4], pfmt(fit_aov[1, 5])))
cat(sprintf("Kruskal-Wallis  : H = %.2f, p = %s\n", kw$statistic, pfmt(kw$p.value)))
cat(sprintf("Levene equal-variance test p = %.5f\n", leveneTest(pActivity ~ series, ser)[1, 3]))
print(ser[, .(n = .N, mean = mean(pActivity), sd = sd(pActivity), median = median(pActivity),
              shapiro_p = shapiro.test(pActivity)$p.value), by = series])



<p class="caption"><b>Figure 3.</b> Potency distributions of the five largest scaffold series (diamonds: group means). The series differ by more than a log unit in median potency, with visibly different spreads and sample sizes (207, 59, 43, 38 and 33 compounds). Both the ANOVA (F = 31.87) and the Kruskal–Wallis test (H = 84.08) reject the hypothesis that all five series have the same central potency, with p-values of 7 × 10<sup>−23</sup> and 2 × 10<sup>−17</sup>, and they agree despite the ANOVA's equal-variance assumption being violated (Levene p below 10<sup>−4</sup>) — a reminder that these tests are fairly robust when sample sizes are large, and that the <em>consequence</em> of a violation should be checked rather than assumed fatal. Note also that Shapiro–Wilk rejects normality for the largest series (p = 0.005) and accepts it for the four small ones, which is the sample-size artefact of <a href="../U7_Estimation/index.html">Unit 7</a> §5, not evidence that the large series is the problematic one.</p>

An omnibus test tells you only that the groups are not all the same. It does not say which pair differs, and running all ten pairwise comparisons afterwards reintroduces exactly the multiplicity problem of §6. The disciplined sequence is: pre-specify the contrast of interest; if there is no such contrast, run the omnibus test first and correct the follow-ups.

For categorical data the same logic applies to a contingency table. The chi-square test compares observed counts with the counts expected if the two classifications were independent; Fisher's exact test computes the same probability exactly from the hypergeometric distribution, which is what you need when any expected count is small.



In [ ]:
cut_off  <- quantile(s, 0.90)                       ## call the top 10 % of the library a 'hit'
dec_rows <- lib[keep][y == 0]
tbl <- table(decoy_set = dec_rows$decoy_set, flagged = s[y == 0] >= cut_off)
ch  <- chisq.test(tbl); fi <- fisher.test(tbl)
cat(sprintf("score cut-off (90th percentile of the library): %.3f\n", cut_off))
print(tbl)
cat("\nexpected counts under independence:\n"); print(round(ch$expected, 1))
cat(sprintf("\nchi-square = %.2f, df = %d, p = %s\n", ch$statistic, ch$parameter, pfmt(ch$p.value)))
cat(sprintf("Fisher exact odds ratio (as the table is given) = %.3f, p = %s\n",
            fi$estimate, pfmt(fi$p.value)))
cat(sprintf("false-positive rate: property-matched %.2f%% | random %.2f%%\n",
            100 * tbl[1, 2] / sum(tbl[1, ]), 100 * tbl[2, 2] / sum(tbl[2, ])))



The two tests agree closely, as they must when every expected count is comfortably above five (the smallest expected count here is 56.4); the exact test matters when a cell contains a handful of observations, which is the common situation in a small clinical table. Note the orientation trap: `fisher.test` returns 0.274 because it takes the table as given — rows ordered `property_matched`, `random` and columns `FALSE`, `TRUE` — and the quantity we want, the odds of being flagged for a matched decoy relative to a random one, is its reciprocal, 3.66. Always state which way round an odds ratio is defined. The odds ratio is the effect size for a 2 × 2 table, and it is the number that belongs in the text — "property-matched decoys were flagged three and a half times as often as random ones" is informative in a way that "χ² = 33.9, p = 6 × 10<sup>−9</sup>" is not.

## 4. What a p-value is, and the four things it is not

A p-value is the probability of data at least as extreme as the observed data, *computed under the assumption that the null hypothesis and the whole statistical model are true*<sup class="c"><a href="#r9">[9]</a></sup><sup class="c"><a href="#r10">[10]</a></sup>. From that definition, four common readings are wrong:

- **It is not the probability that the null hypothesis is true.** That quantity requires a prior and is computed in <a href="../U10_Bayesian/index.html">Unit 10</a>.
- **It is not the probability that the result will replicate.** A study with p = 0.05 has roughly even odds of reaching p &lt; 0.05 again<sup class="c"><a href="#r11">[11]</a></sup>.
- **It is not a measure of effect size.** With 12,000 molecules a difference of 0.001 in mean score reaches p below 10<sup>−10</sup>; with three replicas a difference of 0.1 nm cannot reach 0.05 at all.
- **It is not a decision.** The 0.05 threshold is a convention, not a property of nature, and dichotomising a continuous measure of evidence at it is the practice a large part of the statistical community now argues against<sup class="c"><a href="#r12">[12]</a></sup><sup class="c"><a href="#r13">[13]</a></sup>.

The constructive statement is the relationship between the test and the interval: **a two-sided test at level α rejects exactly when the (1 − α) confidence interval for the effect excludes the null value**. The interval therefore contains the p-value's information plus the effect size plus the precision, which is why reporting the interval instead of the p-value loses nothing and gains everything.



In [ ]:
set.seed(13)
pci <- rbindlist(lapply(c(10, 25, 60, 150, 400), function(n) {
  sa <- sample(act, n); sd_ <- sample(dec, n)
  tt <- t.test(sa, sd_)
  data.table(n_per_group = n, difference = mean(sa) - mean(sd_),
             ci_low = tt$conf.int[1], ci_high = tt$conf.int[2], p = tt$p.value,
             ci_excludes_0 = !(tt$conf.int[1] <= 0 & 0 <= tt$conf.int[2]),
             p_below_0.05 = tt$p.value < 0.05)
}))
p <- ggplot(pci, aes(difference, factor(n_per_group))) +
  geom_errorbarh(aes(xmin = ci_low, xmax = ci_high), height = 0.2, colour = PAL["blue"]) +
  geom_point(colour = PAL["blue"], size = 1.8) +
  geom_vline(xintercept = 0, colour = PAL["orange"], linetype = 2) +
  geom_text(aes(x = ci_high, label = sprintf("p = %.2g", p)), hjust = -0.1, size = 2.6) +
  coord_cartesian(xlim = c(-0.02, 0.42)) +
  labs(x = "difference in mean similarity score (actives - decoys), 95 % CI",
       y = "compounds per group", title = "Every p-value is a confidence interval seen through a keyhole")
print(p)
knitr::kable(pci, digits = 4)



<p class="caption"><b>Figure 4.</b> The same comparison at five sample sizes. The point estimate hardly changes — it ranges from 0.214 to 0.283, because the effect is what it is — while the interval narrows as √n, from a width of 0.25 at n = 10 to 0.033 at n = 400, and the p-value falls from 3.6 × 10<sup>−3</sup> to 6 × 10<sup>−109</sup>. In every row the verdict "p &lt; 0.05" coincides exactly with "the interval excludes zero", as the last two columns of the table confirm. The interval additionally shows that at n = 10 the data are compatible with differences between 0.089 and 0.339, information that no p-value carries.</p>

## 5. Effect sizes, and their intervals

An effect size is a number that answers "how big?" in units that transfer between studies<sup class="c"><a href="#r14">[14]</a></sup>. The ones we need:

- **Cohen's d** = (mean₁ − mean₂)/s<sub>pooled</sub>, the difference in pooled standard deviations.
- **Hedges' g** = d × (1 − 3/(4(n₁+n₂) − 9)), a small-sample bias correction<sup class="c"><a href="#r15">[15]</a></sup>.
- **Cliff's delta** = P(X &gt; Y) − P(X &lt; Y), a non-parametric dominance measure in [−1, 1]<sup class="c"><a href="#r16">[16]</a></sup>.
- **Rank-biserial correlation** = 2U/(n₁n₂) − 1, the same quantity derived from the Mann–Whitney statistic.
- **Odds ratio** for 2 × 2 tables.

In R these live in the `effectsize` package<sup class="c"><a href="#r35">[35]</a></sup>, which returns each one with a confidence interval by default — analytic for d and g, and from the rank statistic for the dominance measures.

### 5.1 Cohen's d traced by hand

Take the first five pActivity values of each of the two largest scaffold series: series 1 gives 7.52, 8.10, 7.00, 6.70 and 7.40; series 2 gives 5.70, 6.23, 7.00, 5.52 and 6.10 (the code below uses the real full series; these excerpts are for the hand calculation).

Mean₁ = (7.52 + 8.10 + 7.00 + 6.70 + 7.40)/5 = 36.72/5 = 7.344. Mean₂ = (5.70 + 6.23 + 7.00 + 5.52 + 6.10)/5 = 30.55/5 = 6.110.

Deviations from mean₁: 0.176, 0.756, −0.344, −0.644, 0.056 → squares 0.0310 + 0.5715 + 0.1183 + 0.4147 + 0.0031 = 1.1386, so s₁² = 1.1386/4 = 0.2847.
Deviations from mean₂: −0.410, 0.120, 0.890, −0.590, −0.010 → squares 0.1681 + 0.0144 + 0.7921 + 0.3481 + 0.0001 = 1.3228, so s₂² = 1.3228/4 = 0.3307.

Pooled variance = ((5−1)·0.2847 + (5−1)·0.3307)/(5 + 5 − 2) = (1.1386 + 1.3228)/8 = 0.3077, so s<sub>pooled</sub> = 0.5547.

**d = (7.344 − 6.110)/0.5547 = 2.225** — a very large effect by any convention (0.2 small, 0.5 medium, 0.8 large). The next chunk reproduces this and then computes every effect size, with its interval, on the full series.



In [ ]:
h1 <- c(7.52, 8.10, 7.00, 6.70, 7.40); h2 <- c(5.70, 6.23, 7.00, 5.52, 6.10)
sp <- sqrt(((length(h1) - 1) * var(h1) + (length(h2) - 1) * var(h2)) / (length(h1) + length(h2) - 2))
cat(sprintf("hand check: mean1 %.3f, mean2 %.3f, s_pooled %.4f, Cohen d = %.3f\n",
            mean(h1), mean(h2), sp, (mean(h1) - mean(h2)) / sp))
s1 <- exact[scaffold_smiles == names(cnt)[1], pActivity]
s2 <- exact[scaffold_smiles == names(cnt)[2], pActivity]
es <- rbindlist(list(
  cbind(data.table(effect_size = "Cohen d"),        as.data.table(cohens_d(s1, s2))[, .(estimate = Cohens_d, CI_low, CI_high)]),
  cbind(data.table(effect_size = "Hedges g"),       as.data.table(hedges_g(s1, s2))[, .(estimate = Hedges_g, CI_low, CI_high)]),
  cbind(data.table(effect_size = "Cliff delta"),    as.data.table(cliffs_delta(s1, s2))[, .(estimate = r_rank_biserial, CI_low, CI_high)]),
  cbind(data.table(effect_size = "rank-biserial r"), as.data.table(rank_biserial(s1, s2))[, .(estimate = r_rank_biserial, CI_low, CI_high)])))
p <- ggplot(es, aes(estimate, factor(effect_size, levels = rev(effect_size)))) +
  geom_errorbarh(aes(xmin = CI_low, xmax = CI_high), height = 0.2, colour = PAL["blue"]) +
  geom_point(colour = PAL["blue"], size = 1.9) +
  geom_vline(xintercept = 0, colour = PAL["orange"], linetype = 2) +
  labs(x = "effect size, series 1 vs series 2 (95 % CI)", y = NULL,
       title = "Four effect sizes for one comparison")
print(p)
cat(sprintf("series 1: n = %d, mean %.3f | series 2: n = %d, mean %.3f | difference %.3f log units\n",
            length(s1), mean(s1), length(s2), mean(s2), mean(s1) - mean(s2)))
cat(sprintf("P(a random series-1 compound is more potent than a random series-2 compound) = %.3f\n",
            mean(outer(s1, s2, ">"))))
knitr::kable(es, digits = 3)



<p class="caption"><b>Figure 5.</b> Four effect sizes for the same comparison between the two largest scaffold series, with 95 % intervals. Cohen's d (1.547) and Hedges' g (1.543) are nearly identical because the samples are large — the correction factor is 0.3 % — and both say the two series differ by about one and a half pooled standard deviations. Cliff's delta and the rank-biserial correlation are <em>exactly</em> the same number here (0.694, interval 0.596 to 0.771) because <code>effectsize</code> implements Cliff's delta as the rank-biserial correlation, of which it is an algebraic rearrangement; the direct computation below the figure confirms the interpretation: a randomly chosen series-1 compound is more potent than a randomly chosen series-2 compound 84.7 % of the time. None of the intervals comes close to zero.</p>

For a 2 × 2 table the matching effect size is the **odds ratio**, and its interval is computed on the log scale, where the standard error is the square root of the sum of the reciprocals of the four cells.



In [ ]:
a_ <- tbl[1, 2]; b_ <- tbl[1, 1]      ## property-matched: flagged, not flagged
c_ <- tbl[2, 2]; d_ <- tbl[2, 1]      ## random decoys:    flagged, not flagged
or_hat <- (a_ * d_) / (b_ * c_)
se_log <- sqrt(1 / a_ + 1 / b_ + 1 / c_ + 1 / d_)
ci <- exp(log(or_hat) + c(-1.96, 1.96) * se_log)
cat(sprintf("property-matched decoys flagged: %d / %d (%.2f %%)\n", a_, a_ + b_, 100 * a_ / (a_ + b_)))
cat(sprintf("random decoys flagged          : %d / %d (%.2f %%)\n", c_, c_ + d_, 100 * c_ / (c_ + d_)))
cat(sprintf("odds ratio %.2f, 95%% CI [%.2f, %.2f]  (log-scale SE %.3f)\n", or_hat, ci[1], ci[2], se_log))
cat(sprintf("risk ratio %.2f  <- the easier quantity to explain in a paper\n",
            (a_ / (a_ + b_)) / (c_ / (c_ + d_))))
cat(sprintf("Fisher exact p = %s | fisher.test's own CI on its orientation: [%.3f, %.3f]\n",
            pfmt(fi$p.value), fi$conf.int[1], fi$conf.int[2]))



The interval excludes 1 comfortably, so the enrichment of property-matched decoys among the false hits is not a fluctuation: a similarity score given decoys chosen to resemble the actives physically produces 2.01 % false positives against 0.56 % for random decoys — an odds ratio of 3.66 (95 % CI 2.31 to 5.79) and a risk ratio of 3.60. That is a statement about the *benchmark*, not about the method, and it is why Unit 19 insists that an enrichment number is meaningless without the decoy-design description. Note that `fisher.test` reports the interval for the reciprocal orientation, which is the same statement upside down.

The practical advice is to report **one** effect size, chosen to match the test, with its interval: d or g alongside a t-test, Cliff's delta or the rank-biserial correlation alongside a rank test, an odds ratio alongside a contingency table. Then translate it into the units of the field — "1.33 log units, i.e. a twenty-fold difference in potency" — because that is the sentence a chemist or a clinician can act on.

<div class="exercise"><h4>Exercise 1</h4>
<p>Compute Cohen's d and the rank-biserial correlation for the comparison of <code>MolWt</code> between the actives and the property-matched decoys of the screening library, and then between the actives and the random decoys (<code>lib[label == 1, MolWt]</code> versus <code>lib[decoy_set == "property_matched", MolWt]</code>). Which decoy set is better matched, and how does the effect size you obtain change the interpretation of a screening benchmark run against each? (Unit 19 develops this into the artificial-enrichment diagnostic.)</p></div>

## 6. Multiple comparisons

Run one test at α = 0.05 on data with no effect and you have a 5 % chance of a false positive. Run 1,000 such tests and you expect 50 of them. Any experiment that measures many things at once — a transcriptome, a descriptor panel, a docking campaign over many targets — is in this regime by construction, and the correction is not optional.

Two different guarantees are on offer, and confusing them is the usual mistake:

- **Family-wise error rate (FWER)** — the probability of *at least one* false positive among all tests. Controlled by **Bonferroni** (reject when p ≤ α/m) and, more powerfully and with the same guarantee, by **Holm's** step-down procedure<sup class="c"><a href="#r17">[17]</a></sup>. Appropriate when a single false claim is costly: one confirmatory endpoint, one go/no-go decision.
- **False discovery rate (FDR)** — the expected *proportion* of false positives among the tests you call significant. Controlled by **Benjamini–Hochberg (BH)**<sup class="c"><a href="#r18">[18]</a></sup>. Appropriate for screening: a gene list or a hit list is a set of leads, and tolerating 5 % wrong leads to find many more true ones is a sensible trade.

All three are `p.adjust(p, method = ...)` in base R — `"bonferroni"`, `"holm"`, `"BH"` — which returns adjusted p-values rather than a reject/accept vector, so the comparison with α happens afterwards. The BH procedure itself is three lines: sort the m p-values ascending, find the largest rank k with \( p_{(k)} \le \frac{k}{m}\alpha \), and reject all hypotheses up to that rank. The corresponding adjusted p-value (q-value in the loose sense) is the running minimum of \( \frac{m}{k}p_{(k)} \) from the largest rank downwards. Storey's refinement additionally estimates the proportion \( \pi_0 \) of true nulls and gains power when it is well below 1<sup class="c"><a href="#r19">[19]</a></sup>.



In [ ]:
set.seed(0)
sel <- sort(sample(nrow(expr), 1000))
Xd  <- as.matrix(expr[sel, dis$sample_id, with = FALSE])    ## 1000 genes x 69 diseased samples
Xh  <- as.matrix(expr[sel, hea$sample_id, with = FALSE])    ## the matching healthy samples
Dm  <- Xd - Xh; npair <- ncol(Dm)
t_paired <- rowMeans(Dm) / (apply(Dm, 1, sd) / sqrt(npair))
p_paired <- 2 * pt(-abs(t_paired), df = npair - 1)
v1 <- apply(Xd, 1, var); v2 <- apply(Xh, 1, var)
se_u <- sqrt(v1 / npair + v2 / npair)
df_u <- (v1 / npair + v2 / npair)^2 /
        ((v1 / npair)^2 / (npair - 1) + (v2 / npair)^2 / (npair - 1))
p_unpaired <- 2 * pt(-abs((rowMeans(Xd) - rowMeans(Xh)) / se_u), df = df_u)
stopifnot(abs(p_paired[1] - t.test(Xd[1, ], Xh[1, ], paired = TRUE)$p.value) < 1e-12)
log2fc <- rowMeans(Dm)
summary_dt <- rbindlist(lapply(list(`paired t` = p_paired, `unpaired Welch t` = p_unpaired), function(p)
  data.table(raw = sum(p < 0.05), Bonferroni = sum(p.adjust(p, "bonferroni") < 0.05),
             Holm = sum(p.adjust(p, "holm") < 0.05), BH = sum(p.adjust(p, "BH") < 0.05))), idcol = "design")
hist_dt <- rbind(data.table(p = p_paired, design = "paired t"),
                 data.table(p = p_unpaired, design = "unpaired Welch t"))
p <- ggplot(hist_dt, aes(p)) +
  geom_histogram(bins = 40, fill = PAL["blue"], colour = NA) +
  geom_hline(yintercept = 1000 / 40, colour = PAL["orange"], linetype = 2) +
  facet_wrap(~ design) +
  labs(x = "p-value", y = "genes", title = "p-value histograms: a spike at zero on a flat background")
print(p)
cat(sprintf("genes tested: %d | mean |log2 fold change| = %.3f | genes with |log2FC| > 1: %d\n",
            length(p_paired), mean(abs(log2fc)), sum(abs(log2fc) > 1)))
knitr::kable(summary_dt)



<p class="caption"><b>Figure 6.</b> Histograms of the 1,000 p-values under the two designs; the dashed line is the height a uniform histogram would have if every null were true. Neither is uniform: both show the enormous spike near zero characteristic of a real, pervasive biological signal — periodontitis changes the expression of a large fraction of the transcriptome in affected tissue. The paired design (left) pushes more mass into the first bin than the unpaired one (right). The table gives the counts: of 1,000 genes, 756 reach raw p &lt; 0.05 under pairing against 701 unpaired, and after Benjamini–Hochberg correction 747 versus 682 survive. The family-wise procedures are far stricter: under pairing they keep 534 (Bonferroni) and 551 (Holm), and without pairing 453 and 470 — Holm rejecting slightly more than Bonferroni in both designs, as it must.</p>

Three readings follow. First, **the design matters more than the correction**: pairing recovers 65 more genes after BH than the unpaired analysis does, and no choice of adjustment method can make up for the lost information. Second, **BH costs almost nothing here and Bonferroni costs a lot** — when the signal is pervasive, controlling the probability of a single error discards more than two hundred true findings. Third, these counts are so high only because the biological effect is strong; in a typical study the first bin is a modest excess over a uniform background, and then the choice of procedure decides whether the paper has ten hits or none.

<div class="box warn"><h4>The p-value histogram is a diagnostic, not decoration</h4>
<p>Always plot it before correcting. A <strong>uniform</strong> histogram means no signal. A <strong>spike at zero on a flat background</strong> is the healthy pattern. A histogram that rises towards 1, or is bumpy in the middle, means the tests are misspecified — wrong null, dependence between tests, or discrete data — and no multiplicity correction repairs a wrong null.</p></div>



In [ ]:
bh_by_hand <- function(p, alpha = 0.05) {
  m <- length(p); o <- order(p); ps <- p[o]
  below <- which(ps <= alpha * seq_len(m) / m)
  k <- if (length(below)) max(below) else 0
  q <- rev(cummin(rev(m / seq_len(m) * ps)))        ## running minimum from the largest rank down
  qv <- numeric(m); qv[o] <- pmin(q, 1)
  list(k = k, q = qv)
}
bh <- bh_by_hand(p_paired, 0.05)
cat(sprintf("BH by hand rejects %d | p.adjust rejects %d | identical adjusted p-values: %s\n",
            bh$k, sum(p.adjust(p_paired, "BH") < 0.05),
            isTRUE(all.equal(bh$q, p.adjust(p_paired, "BH")))))
cat(sprintf("BH cut-off: the %dth smallest p-value, p = %.3e (Bonferroni cut-off would be %.3e)\n",
            bh$k, sort(p_paired)[bh$k], 0.05 / length(p_paired)))
pi0 <- min(1, 2 * mean(p_paired > 0.5))
cat(sprintf("Storey pi0 estimate (fraction of true nulls): %.3f -> BH is conservative by a factor %.1f here\n",
            pi0, 1 / pi0))
srt <- sort(p_paired); m <- length(srt)
p <- ggplot(data.frame(rank = seq_len(m), p = srt), aes(rank, p)) +
  geom_point(size = 0.4, colour = PAL["blue"]) +
  geom_line(data = data.frame(rank = seq_len(m), p = 0.05 * seq_len(m) / m), colour = PAL["orange"]) +
  geom_hline(yintercept = 0.05 / m, linetype = 3) +
  geom_vline(xintercept = bh$k, colour = PAL["green"], linetype = 2) +
  scale_x_log10() + scale_y_log10() +
  labs(x = "rank k", y = "p-value",
       title = sprintf("The Benjamini-Hochberg staircase (rejects the first %d)", bh$k))
print(p)



<p class="caption"><b>Figure 7.</b> The BH procedure drawn on log–log axes. The blue points are the 1,000 sorted p-values; the orange line is the BH threshold k·α/m; the procedure rejects everything up to the last crossing, marked by the green dashed line at rank 747, which corresponds to a p-value cut-off of 0.036 — seven hundred times more permissive than the Bonferroni cut-off of 5 × 10<sup>−5</sup> (dotted line). The printed Storey estimate of the proportion of true nulls is 0.124, so BH is conservative by a factor of eight on these data: the adaptive version would reject slightly more.</p>

The hand implementation and `p.adjust` return identical adjusted p-values, which is worth checking once in your life so that the function stops being a black box. Report the adjusted values in the supplementary table, state the method and the α in the methods section, and say how many tests were in the family — a BH correction over 1,000 genes means nothing if the reader does not know whether you tested 1,000 or 20,000.

### 6.1 A negative control: what the procedure does when nothing is there

The only convincing demonstration that a multiplicity correction works is to run it where the truth is known. We take only the 69 **healthy** samples, split them at random into two groups, and test all 1,000 genes — there is no disease effect to find, because every sample is healthy.



In [ ]:
H <- as.matrix(expr[sel, hea$sample_id, with = FALSE])
set.seed(7)
pm <- sample(ncol(H)); g1 <- pm[1:34]; g2 <- pm[35:69]
v1 <- apply(H[, g1], 1, var); v2 <- apply(H[, g2], 1, var)
se_n <- sqrt(v1 / 34 + v2 / 35)
df_n <- (v1 / 34 + v2 / 35)^2 / ((v1 / 34)^2 / 33 + (v2 / 35)^2 / 34)
p_null <- 2 * pt(-abs((rowMeans(H[, g1]) - rowMeans(H[, g2])) / se_n), df = df_n)
cmp <- rbind(data.table(p = p_null,   what = "healthy vs healthy: flat"),
             data.table(p = p_paired, what = "diseased vs healthy: spike at zero"))
p <- ggplot(cmp, aes(p)) +
  geom_histogram(bins = 40, fill = "grey70", colour = NA) +
  geom_hline(yintercept = 1000 / 40, colour = PAL["orange"], linetype = 2) +
  facet_wrap(~ what, scales = "free_y") +
  labs(x = "p-value", y = "genes", title = "A null comparison validates the whole pipeline")
print(p)
cat("NULL comparison (69 healthy samples split 34 / 35):\n")
cat(sprintf("  raw p < 0.05        : %3d genes   (expected by chance: ~50)\n", sum(p_null < 0.05)))
cat(sprintf("  Bonferroni          : %3d genes\n", sum(p.adjust(p_null, "bonferroni") < 0.05)))
cat(sprintf("  Benjamini-Hochberg  : %3d genes\n", sum(p.adjust(p_null, "BH") < 0.05)))
cat(sprintf("  smallest p-value    : %.4f\n", min(p_null)))
cat("REAL comparison (paired, diseased vs healthy):\n")
cat(sprintf("  raw p < 0.05        : %3d genes\n", sum(p_paired < 0.05)))
cat(sprintf("  Benjamini-Hochberg  : %3d genes\n", sum(p.adjust(p_paired, "BH") < 0.05)))



<p class="caption"><b>Figure 8.</b> The same 1,000 genes under a null comparison (left, healthy against healthy) and the real one (right); note the free y-axis. The null histogram is flat, as a correct test on true nulls must be, and the 8 genes that reach raw p &lt; 0.05 — fewer than the ~50 expected, because this split is itself one realisation — are precisely the false positives multiplicity creates. Not one of them survives Benjamini–Hochberg or Bonferroni, and the smallest p-value in the whole family is 0.0089. The real comparison (right) is a different object entirely.</p>

This negative control costs three lines and should be run in every high-dimensional analysis in a thesis. If the null histogram is not flat, the test is wrong before any correction is applied — a hidden batch structure, a dependence the test ignores, or the wrong exchangeability assumption.

## 7. How significance is manufactured: p-hacking, forking paths and HARKing

None of the machinery above protects against the most common source of false findings, which is not a wrong formula but an **unregistered analytic freedom**. Three named versions:

- **p-hacking**: trying analyses until one is significant — dropping outliers, adding a covariate, switching tests, collecting more data until p dips below 0.05<sup class="c"><a href="#r20">[20]</a></sup>. The published literature shows the footprint of this practice as an excess of p-values just below 0.05<sup class="c"><a href="#r21">[21]</a></sup>.
- **The garden of forking paths**<sup class="c"><a href="#r22">[22]</a></sup>: the same inflation without any dishonesty, because the analysis choices were made *after* seeing the data, so the test that was eventually run was one of many that could have been run.
- **HARKing**<sup class="c"><a href="#r23">[23]</a></sup>: presenting a hypothesis formulated after the results as if it had been stated in advance, which converts an exploratory finding into a confirmatory claim it has not earned.

The simulation below quantifies the second one on a dataset in which, by construction, there is nothing to find.



In [ ]:
forking_paths <- function(seed) {
  set.seed(seed)
  x <- rnorm(60); grp <- rep(0:1, each = 30); cov <- rnorm(60)   ## no effect whatsoever
  c(t.test(x[grp == 0], x[grp == 1], var.equal = TRUE)$p.value,            ## 1 plain t
    t.test(x[grp == 0], x[grp == 1])$p.value,                             ## 2 Welch
    suppressWarnings(wilcox.test(x[grp == 0], x[grp == 1])$p.value),      ## 3 rank test
    { k <- abs(x - mean(x)) < 2 * sd(x)                                   ## 4 drop 'outliers'
      t.test(x[k & grp == 0], x[k & grp == 1])$p.value },
    t.test(x[grp == 0 & cov > 0],  x[grp == 1 & cov > 0])$p.value,        ## 5 subgroup
    t.test(x[grp == 0 & cov <= 0], x[grp == 1 & cov <= 0])$p.value,       ## 6 other subgroup
    t.test(log1p(abs(x[grp == 0])), log1p(abs(x[grp == 1])))$p.value,     ## 7 transform
    t.test(x[grp == 0][1:20], x[grp == 1][1:20])$p.value,                 ## 8 stop early
    var.test(x[grp == 0], x[grp == 1])$p.value,                           ## 9 test the variance
    suppressWarnings(ks.test(x[grp == 0], x[grp == 1])$p.value))          ## 10 test the distribution
}
P <- t(sapply(1:2000, forking_paths))
best <- apply(P, 1, min); single <- P[, 1]
dd <- rbind(data.table(p = single, what = "pre-specified test only"),
            data.table(p = best,   what = "smallest of 10 analyses"))
p <- ggplot(dd, aes(p, fill = what)) +
  geom_histogram(bins = 30, alpha = 0.7, position = "identity") +
  geom_vline(xintercept = 0.05, linetype = 2) +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "p-value reported", y = "simulated studies",
       title = "Ten defensible choices turn a 5 % error rate into something else")
print(p)
cat("true effect in these data: ZERO\n")
cat(sprintf("false-positive rate, pre-specified single test : %.1f %%\n", 100 * mean(single < 0.05)))
cat(sprintf("false-positive rate, reporting the best of 10  : %.1f %%\n", 100 * mean(best < 0.05)))
cat(sprintf("median reported p-value, best of 10            : %.3f\n", median(best)))



<p class="caption"><b>Figure 9.</b> Distribution of the reported p-value over 2,000 simulated studies in which the true effect is exactly zero. Reporting a single pre-specified test (blue) gives the correct error rate, 4.4 % here. Reporting the smallest p-value among ten analyses that any referee would accept as reasonable (orange) raises it to 22.4 % — one study in four or five — and no individual step in that process is fraudulent; the median reported p-value falls to 0.135. The forking-path inflation is a property of the <em>procedure</em>, which is why only pre-specification or an explicit multiplicity correction over the whole garden can control it.</p>

<div class="box key"><h4>A pre-registered analysis plan for a thesis chapter</h4>
<p>Written <em>before</em> the data are looked at, half a page, in the thesis repository with a commit date:</p>
<ol>
<li><b>Question and primary endpoint.</b> One sentence, one number. "Does scaffold series A have higher median pActivity than series B?"</li>
<li><b>Data and exclusions.</b> Which rows, which filters, and the exclusion rules — decided now, not after seeing which exclusions help.</li>
<li><b>Primary analysis.</b> The test, the effect size, the α, one-sided or two-sided, and the software.</li>
<li><b>Multiplicity.</b> How many hypotheses belong to the family, and which correction.</li>
<li><b>Sample size / power.</b> The effect size worth detecting and the n or number of replicas it requires (§8).</li>
<li><b>Everything else is exploratory</b>, and will be labelled as such in the text. Exploratory findings are hypotheses for the next experiment, not conclusions.</li>
</ol>
<p>This is cheap, it protects the student rather than constraining them, and the practice is now standard in several fields<sup class="c"><a href="#r24">[24]</a></sup><sup class="c"><a href="#r25">[25]</a></sup>.</p></div>

## 8. Power: deciding how many replicas or compounds you need

**Power** is the probability that the test rejects the null when a specified effect really exists. It depends on the effect size, the variability, the sample size and α — fix any three and the fourth follows. A study with low power is not merely likely to miss a real effect; its *significant* results are also less likely to be true and are systematically exaggerated<sup class="c"><a href="#r26">[26]</a></sup><sup class="c"><a href="#r27">[27]</a></sup>.

Closed-form power formulas exist for the t-test — `power.t.test()` in base R — but simulation is more general and far harder to get wrong: generate data under the alternative you care about, run your actual analysis pipeline, count the rejections.



In [ ]:
sd_between <- sd(c(A - mean(A), B - mean(B)))       ## between-replica SD at fixed temperature
power_sim <- function(n, delta, sdv, nsim = 4000, alpha = 0.05, seed = 0) {
  set.seed(seed)
  x <- matrix(rnorm(nsim * n, 0, sdv), nrow = nsim)
  z <- matrix(rnorm(nsim * n, delta, sdv), nrow = nsim)
  v1 <- apply(x, 1, var); v2 <- apply(z, 1, var)
  se <- sqrt(v1 / n + v2 / n)
  dfw <- (v1 / n + v2 / n)^2 / ((v1 / n)^2 / (n - 1) + (v2 / n)^2 / (n - 1))
  mean(2 * pt(-abs((rowMeans(x) - rowMeans(z)) / se), df = dfw) < alpha)
}
deltas <- seq(0, 0.20, by = 0.01)
curves <- rbindlist(lapply(c(3, 5, 10, 20), function(n)
  data.table(n = factor(n), delta = deltas,
             power = sapply(deltas, function(d) power_sim(n, d, sd_between, seed = n)))))
p <- ggplot(curves, aes(delta, power, colour = n)) +
  geom_line() +
  geom_hline(yintercept = 0.8, linetype = 2) +
  geom_vline(xintercept = abs(mean(A) - mean(B)), linetype = 3) +
  scale_colour_manual(values = unname(PAL), name = "replicas\nper condition") +
  labs(x = "true difference in mean C-alpha RMSD (nm)", y = "power at alpha = 0.05",
       title = "Three replicas cannot see anything smaller than the replica spread")
print(p)
cat(sprintf("between-replica SD at fixed temperature: %.4f nm | observed difference %.4f nm\n",
            sd_between, abs(mean(A) - mean(B))))
for (n in c(3, 5, 10, 20))
  cat(sprintf("  n = %2d replicas: power for 0.02 nm %.3f | 0.05 nm %.3f | 0.10 nm %.3f | 0.15 nm %.3f\n",
              n, power_sim(n, 0.02, sd_between, seed = n), power_sim(n, 0.05, sd_between, seed = n),
              power_sim(n, 0.10, sd_between, seed = n), power_sim(n, 0.15, sd_between, seed = n)))



<p class="caption"><b>Figure 10.</b> Power curves for the temperature comparison, using the measured between-replica standard deviation of 0.0502 nm. With the three replicas we actually ran, the power to detect the difference we actually observed (dotted vertical line at 0.0203 nm) is 4.2 % — indistinguishable from the false-positive rate, which is another way of saying the experiment was blind to this effect. Three replicas reach 66 % power only at 0.15 nm, seven times the observed difference; ten replicas reach 98.5 % power at 0.10 nm and 55 % at 0.05 nm; twenty replicas reach 86 % at 0.05 nm. At the observed difference of 0.02 nm the four designs reach only 4.2 %, 6.6 %, 13.4 % and 21.8 % power respectively, which is the quantitative version of "this experiment could not have seen it".</p>

This calculation should be run *before* the simulations, not after, and it is the quantitative answer to the referee question "why three replicas?". The honest answer in this dataset is that three replicas were chosen for teaching and for compute budget, and that the correct conclusion from them is "no detectable difference at 1 ns", not "no difference".

The same machinery sizes a chemistry experiment, and here the closed form is enough.



In [ ]:
sd_pool <- sqrt((var(s1) + var(s2)) / 2)
ns <- seq(5, 400, by = 5)
pw <- rbindlist(lapply(c(0.2, 0.3, 0.5, 1.0), function(delta)
  data.table(delta = factor(sprintf("%.1f log units", delta)), n = ns,
             power = sapply(ns, function(n) power.t.test(n = n, delta = delta, sd = sd_pool,
                                                         sig.level = 0.05)$power))))
p <- ggplot(pw, aes(n, power, colour = delta)) +
  geom_line() + geom_hline(yintercept = 0.8, linetype = 2) +
  scale_colour_manual(values = unname(PAL), name = "true difference") +
  labs(x = "compounds per series", y = "power at alpha = 0.05",
       title = "Detecting a potency difference between two series")
print(p)
cat(sprintf("pooled SD of pActivity within a series: %.3f log units\n", sd_pool))
for (delta in c(0.2, 0.3, 0.5, 1.0))
  cat(sprintf("  to detect %.1f log units with 80%% power: %5.0f compounds per series\n",
              delta, ceiling(power.t.test(delta = delta, sd = sd_pool, power = 0.8, sig.level = 0.05)$n)))
cat(sprintf("series actually available: S1 n = %d, S2 n = %d\n", length(s1), length(s2)))



<p class="caption"><b>Figure 11.</b> Power against series size for four true potency differences, at the pooled within-series standard deviation of 0.913 log units measured in our own data. Detecting a 0.3-log difference — roughly a factor of two in IC50, the smallest difference a medicinal chemist would act on — needs 147 compounds per series at 80 % power, and a 0.2-log difference needs 328. Our second-largest series has 59 members, so a comparison involving it is powered only for differences of about half a log unit (54 compounds) and above. A one-log-unit difference needs only 15 compounds per side, which is why "series A is ten times more potent" is a claim small datasets can support and "series A is twice as potent" usually is not.</p>

### 8.1 The winner's curse

Low power does not only cause misses. Among the comparisons that *do* reach significance in an underpowered study, the estimated effects are systematically too large, because only the luckiest samples cross the threshold. This is the **winner's curse**, and it explains a large part of why effect sizes shrink on replication.



In [ ]:
true_delta <- 0.30; sd_w <- 0.90          ## a real 0.3-log difference, within-series SD as measured
wc <- rbindlist(lapply(c(10, 20, 40, 80, 160, 320), function(n) {
  set.seed(n)
  a <- matrix(rnorm(3000 * n, true_delta, sd_w), nrow = 3000)
  b <- matrix(rnorm(3000 * n, 0, sd_w), nrow = 3000)
  est <- rowMeans(a) - rowMeans(b)
  v1 <- apply(a, 1, var); v2 <- apply(b, 1, var); se <- sqrt(v1 / n + v2 / n)
  dfw <- (v1 / n + v2 / n)^2 / ((v1 / n)^2 / (n - 1) + (v2 / n)^2 / (n - 1))
  sig <- 2 * pt(-abs(est / se), df = dfw) < 0.05
  data.table(n_per_group = n, power = mean(sig), all_studies = mean(est),
             significant_only = mean(est[sig]), inflation = mean(est[sig]) / true_delta,
             sign_errors = mean(est[sig] < 0))
}))
p <- ggplot(melt(wc[, .(n_per_group, all_studies, significant_only)], id.vars = "n_per_group"),
            aes(n_per_group, value, colour = variable)) +
  geom_line() + geom_point(size = 1.6) +
  geom_hline(yintercept = true_delta, linetype = 2) +
  scale_x_log10() +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "compounds per series", y = "estimated difference (log units)",
       title = "The winner's curse: small significant studies exaggerate")
print(p)
knitr::kable(wc, digits = 3)



<p class="caption"><b>Figure 12.</b> Simulated studies of a real 0.3-log effect (dashed line). Averaged over all studies (blue) the estimate is unbiased at every sample size, 0.293 to 0.313. Averaged over only the studies that reached p &lt; 0.05 (orange) — the ones that would be published — the estimate is 0.909 at n = 10, an inflation factor of 3.0, and is still inflated by 35 % at n = 80, converging to the truth only once power exceeds 0.8. The last column of the table shows the darker consequence: in the smallest studies 2.8 % of the "significant" results have the <em>wrong sign</em>.</p>

The practical consequences are immediate: never take an effect size from a small significant study as the input to your own power calculation (it is biased upwards, so your study will be underpowered), and treat a surprisingly large effect in a small sample as a reason for replication rather than for celebration.

## 9. The centrepiece: frame level versus replica level

Everything in this unit now converges on one experiment. We ask a single question — *is the C-alpha RMSD of the Trp-cage different at 340 K than at 300 K after 1 ns?* — and answer it twice with the same data: once treating each saved **frame** as an observation (1,500 per condition) and once treating each **replica** as an observation (three per condition).



In [ ]:
frames_A <- md_obs[condition == "A", rmsd_ca_nm]
frames_B <- md_obs[condition == "B", rmsd_ca_nm]
t_frame <- t.test(frames_A, frames_B)
u_frame <- suppressWarnings(wilcox.test(frames_A, frames_B))
t_rep   <- t.test(A, B)
verdict <- data.table(
  unit = c("frame", "frame", "replica", "replica"),
  n_per_group = c(length(frames_A), length(frames_A), 3L, 3L),
  difference_nm = c(rep(mean(frames_A) - mean(frames_B), 2), rep(mean(A) - mean(B), 2)),
  test = c("Welch t", "Mann-Whitney", "Welch t", "permutation (exact, 20 splits)"),
  p = c(pfmt(t_frame$p.value), pfmt(u_frame$p.value), sprintf("%.3f", t_rep$p.value),
        sprintf("%.3f", p_exact)))
md_plot <- copy(md_obs)
md_plot[, xpos := ifelse(condition == "A", 1, 2) + (replica - 2) * 0.22]
set.seed(14)
md_plot[, xjit := xpos + rnorm(.N, 0, 0.02)]
p <- ggplot(md_plot, aes(xjit, rmsd_ca_nm, colour = condition)) +
  geom_point(size = 0.15, alpha = 0.25) +
  geom_point(data = md_plot[, .(xpos = xpos[1], rmsd_ca_nm = mean(rmsd_ca_nm)), by = .(condition, replica)],
             aes(xpos, rmsd_ca_nm), size = 2.6, shape = 21, fill = "white", stroke = 1.1) +
  scale_x_continuous(breaks = c(1, 2), labels = c("300 K (3 replicas)", "340 K (3 replicas)")) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")]), guide = "none") +
  labs(x = NULL, y = "C-alpha RMSD (nm)",
       title = "Frames are many and dependent; replicas are few and independent")
print(p)
cat(sprintf("frame-level  : mean A %.4f vs B %.4f nm, Welch t = %.2f, p = %.3e\n",
            mean(frames_A), mean(frames_B), t_frame$statistic, t_frame$p.value))
cat(sprintf("replica-level: mean A %.4f vs B %.4f nm, Welch t = %.3f, p = %.3f (df %.2f)\n",
            mean(A), mean(B), t_rep$statistic, t_rep$p.value, t_rep$parameter))
cat(sprintf("ratio of p-values: %.1e\n", t_frame$p.value / t_rep$p.value))
knitr::kable(verdict)



<p class="caption"><b>Figure 13.</b> Every frame of all six runs (small dots) with the six replica means as open circles. The two conditions overlap almost completely, and the separation <em>within</em> condition A — whose three replicas average 0.110, 0.160 and 0.252 nm — is larger than the separation between the conditions (0.1736 against 0.1533 nm). The frame-level cloud is dense because frames are plentiful, not because the quantity is precisely known.</p>

The table reports the two verdicts. At frame level the Welch test gives t = 10.81 and p = 1.14 × 10<sup>−26</sup>, and the Mann–Whitney test agrees at 4.8 × 10<sup>−13</sup>; at replica level the same comparison gives t = 0.443 and p = 0.690, and the exact permutation test gives 0.800 and could not have gone below 0.10 whatever happened. The two p-values differ by a factor of 1.7 × 10<sup>−26</sup>. **The frame-level p-value is not a stronger version of the replica-level one; it is an answer to a different and uninteresting question** — "do these 3,000 particular correlated numbers come from the same distribution?" — obtained by treating 3,000 dependent observations as 3,000 independent experiments. This is pseudoreplication<sup class="c"><a href="#r28">[28]</a></sup>, documented as a pervasive error in neuroscience<sup class="c"><a href="#r29">[29]</a></sup> and equally common in simulation papers, where it is produced by the simple act of pasting a trajectory into a t-test.



In [ ]:
obs_cols <- c("temperature_K", "box_volume_nm3", "rg_nm", "n_hbonds", "helix_fraction", "dist_W6_P12_nm")
both <- rbindlist(lapply(obs_cols, function(v) {
  pr <- md_obs[, .(m = mean(get(v))), by = .(condition, replica)]
  a <- pr[condition == "A", m]; b <- pr[condition == "B", m]
  data.table(observable = v, mean_A = mean(a), mean_B = mean(b),
             p_replica = t.test(a, b)$p.value,
             p_frame = t.test(md_obs[condition == "A", get(v)], md_obs[condition == "B", get(v)])$p.value)
}))
both[, holm_replica := p.adjust(p_replica, "holm")]
both[, `:=`(mean_A = signif(mean_A, 5), mean_B = signif(mean_B, 5),
            p_replica = signif(p_replica, 3), p_frame = signif(p_frame, 3),
            holm_replica = signif(holm_replica, 3))]
knitr::kable(both)



Six observables, two units of analysis. The two **control** observables — temperature and box volume — separate the conditions at replica level with p = 5.8 × 10<sup>−6</sup> and 4.6 × 10<sup>−7</sup>, surviving a Holm correction over the six tests, because a thermostat has almost no between-replica variance: the simulations did what they were told. Not one **structural** observable (radius of gyration, hydrogen bonds, helix fraction, cage distance) differs detectably at replica level, with p-values from 0.36 to 0.88. At frame level, by contrast, four of them look overwhelmingly significant — the helix fraction reaches 2.6 × 10<sup>−26</sup> and the cage distance 2.4 × 10<sup>−30</sup> — which is the same illusion as in Figure 13, repeated four times.

Three honest statements can be made about this dataset, and none of them is "340 K destabilises the protein":

1. The thermostat and barostat worked: temperature (300.42 K versus 340.24 K) and box volume (47.76 versus 48.84 nm³) separate the conditions at replica level.
2. No structural observable differs detectably at 1 ns.
3. The design had essentially no power to detect a difference of the size observed (§8), so statement 2 is "not detected", not "not there". Trp-cage unfolds on microsecond timescales, so a 1 ns window is not expected to show it.

<div class="box warn"><h4>The rule for simulation statistics</h4>
<p>Identify the experimental unit before choosing a test. For molecular dynamics it is the <strong>independent run</strong>, not the frame<sup class="c"><a href="#r31">[31]</a></sup>; for a clinical study with several samples per patient it is the <strong>patient</strong>; for a plate-based assay it is often the <strong>plate</strong>. If you have three units you have n = 3, and the correct report is an estimate with its between-unit interval plus an explicit statement of what effect size the design could have detected. If the frames must be used — for example to estimate an equilibrium average — then the autocorrelation correction of <a href="../U7_Estimation/index.html">Unit 7</a> applies, and a mixed-effects model (<a href="../U9_Linear_Models/index.html">Unit 9</a>, <a href="../U10_Bayesian/index.html">Unit 10</a>) is the principled way to use both levels at once.</p></div>

## 10. Choosing a test, in practice

<div class="box key"><h4>Decision sequence</h4>
<ol>
<li><b>What is the experimental unit?</b> Count those, not the measurements. If n is below about 6, prefer an exact permutation test and check whether significance is even attainable.</li>
<li><b>Is the design paired?</b> If there is a natural partner for each observation, use it — the gain usually exceeds anything the choice of test provides.</li>
<li><b>What statistic answers the scientific question?</b> Means, medians, proportions, ranks, an AUC. Write the effect size first, then pick the test that matches it.</li>
<li><b>Check the assumptions that matter.</b> Independence always; equal variances only for Student and ANOVA (R's <code>t.test</code> defaults to Welch, and you should leave it there); normality mainly for small n (for large n the CLT covers means, as Unit 7 showed).</li>
<li><b>Count the family.</b> How many hypotheses could have been tested? Correct with Holm if a single false claim is costly, with BH if the output is a list of leads.</li>
<li><b>Report</b> the effect size with its interval, the n, the test, the correction and the power. The p-value is the least informative of these and should never travel alone<sup class="c"><a href="#r32">[32]</a></sup>.</li>
</ol></div>

The two-page "Points of Significance" columns in *Nature Methods* are an excellent companion to this checklist, in particular those on significance and t-tests<sup class="c"><a href="#r33">[33]</a></sup> and on power and sample size<sup class="c"><a href="#r34">[34]</a></sup>. All computations here used base R's `stats`<sup class="c"><a href="#r1">[1]</a></sup>, `car`<sup class="c"><a href="#r36">[36]</a></sup> and `effectsize`<sup class="c"><a href="#r35">[35]</a></sup>, on data curated from ChEMBL<sup class="c"><a href="#r30">[30]</a></sup> and from a public gingival-tissue expression series<sup class="c"><a href="#r2">[2]</a></sup>.

<div class="exercise"><h4>Exercise 2</h4>
<p>The last chunk tested six observables at both levels. Extend it: add the <b>exact permutation p-value</b> (all 20 labellings) for each observable at replica level, and a <b>Cliff's delta</b> with its interval at frame level. Then answer in writing: which observables separate the conditions at replica level, why exactly those ones, and what the frame-level p-values of the other four are evidence <em>of</em>? Finally, re-run the whole table using only the last 500 ps of each trajectory and say whether any conclusion changes.</p></div>

<div class="box bridge"><h4>Bridge</h4>
<p>This unit sits between <a href="../U7_Estimation/index.html">Unit 7</a>, which built the sampling distributions and intervals that every test here manipulates, and <a href="../U9_Linear_Models/index.html">Unit 9</a>, where a t-test becomes a test on a regression coefficient, an ANOVA becomes a linear model with categorical predictors, and the paired design becomes a blocking factor. <a href="../U10_Bayesian/index.html">Unit 10</a> answers the question a p-value cannot. The Python counterpart of this unit is <a href="../../Python/U8_Hypothesis_Testing/index.html">here</a> and the R edition is indexed at <a href="../index.html">the course root</a>. Forward, <a href="../U12_Validation/index.html">Unit 12</a> applies permutation testing to model metrics as y-randomisation and compares two models with paired tests; <a href="../U20_MD_Analytics/index.html">Unit 20</a> develops the replica statistics of §9 in full; <a href="../U21_Omics/index.html">Unit 21</a> scales §6 to a whole transcriptome with moderated tests. In the sister courses, the bioinformatics unit on <a href="../../../BGM324/05_Molecular_Dynamics/index.html">molecular dynamics</a> produces the trajectories analysed here, the one on <a href="../../../BGM324/03_Molecular_Docking/index.html">molecular docking</a> produces the ranked lists whose enrichment §2 tests, and the cheminformatics unit on <a href="../../../Cheminformatics/U14_Validation/index.html">validation</a> uses the same permutation machinery on model scores.</p></div>

## 11. Summary

You should be able to explain, without notes:

- the four components of a test, and how to build a null distribution by permutation for any statistic;
- why the classical tests are shortcuts of the permutation test, and which assumption each one buys its shortcut with;
- why Welch's test is R's default and a better choice than Student's (here t = 55 against a misleading 137), and why pairing is worth more than any choice of test;
- what a p-value is, the four things it is not, and that a test and a confidence interval are the same statement;
- how to compute and report Cohen's d, Hedges' g, Cliff's delta and an odds ratio with intervals, and how to translate them into the units of the field;
- the difference between family-wise error and false discovery rate, how the BH procedure works line by line, and why a p-value histogram must be inspected before correcting;
- how a null comparison (healthy versus healthy, 8 raw hits and none surviving correction) validates an entire multiplicity pipeline;
- how the garden of forking paths inflates a 4.4 % error rate to 22.4 % without any dishonesty, and what a pre-registered analysis plan contains;
- how to estimate power by simulation, that 147 compounds per series are needed for a 0.3-log difference, and why significant small studies exaggerate by a factor of three;
- why the experimental unit, not the number of measurements, determines n — and that getting it wrong moved a p-value by 26 orders of magnitude.

## 12. Self-assessment

<details><summary>1. Your permutation test with 1,000 shuffles reports p = 0. What do you write in the paper?</summary>"p &lt; 0.001 (1,000 permutations)". A Monte-Carlo p-value is computed as (1 + number of shuffles at least as extreme)/(B + 1), so it can never legitimately be zero; the resolution of the test is 1/(B + 1), which was 0.0002 for the 5,000 permutations of §2.</details>

<details><summary>2. Three replicas per condition. What is the smallest two-sided p-value an exact permutation test can produce?</summary>2/20 = 0.10, because there are only choose(6,3) = 20 ways to label six values as 3 + 3 and the observed labelling plus its mirror image are always at least as extreme as themselves. The design cannot reach 0.05 no matter what the data show.</details>

<details><summary>3. When should you use Student's t-test rather than Welch's?</summary>Almost never, which is why <code>t.test()</code> in R defaults to Welch and you must write <code>var.equal = TRUE</code> to get Student's. Welch's test costs essentially nothing in power when the variances are equal and protects the error rate when they are not, particularly with unequal group sizes — in §3 the variance ratio was 24 and the two statistics differed by a factor of 2.5.</details>

<details><summary>4. A referee asks you to "add a non-parametric test because the data are not normal". Is that a good reason?</summary>Partly. Rank tests make fewer distributional assumptions but they answer a different question (stochastic ordering, not a difference of means) and they are not assumption-free — they still require independence, and the Wilcoxon signed-rank test assumes symmetric differences. With large samples the t-test on means is robust by the CLT; with small samples a permutation test on the statistic you actually care about is usually the better answer.</details>

<details><summary>5. You test 1,000 genes and 50 reach p &lt; 0.05. Have you found anything?</summary>Not yet: 50 is exactly what 1,000 true nulls would produce. Look at the p-value histogram (flat means nothing there), apply BH, and check whether any gene survives. In the null experiment of §6.1 eight genes reached raw p &lt; 0.05 and none survived correction.</details>

<details><summary>6. What is the difference between controlling FWER and controlling FDR?</summary>FWER is the probability of making at least one false rejection in the whole family; FDR is the expected proportion of false rejections among the rejections made. Controlling FWER is stricter and appropriate for confirmatory claims; controlling FDR is appropriate when the output is a ranked list of candidates for follow-up. Here BH kept 747 genes and Bonferroni 534.</details>

<details><summary>7. Your paired analysis gives p = 10⁻²⁰ and the unpaired analysis of the same data gives p = 10⁻¹⁷. Which do you report?</summary>The paired one, because it matches the design: each diseased sample has a healthy partner from the same patient, and the unpaired test wastes that information by letting between-patient variation into the comparison. Report the design in the methods; a reader must be able to tell which test applies.</details>

<details><summary>8. Why is a result from a small, significant study a bad basis for a power calculation?</summary>Because of the winner's curse: among underpowered studies, only the ones whose estimate was luckily large reach significance, so the published effect is biased upwards — by a factor of 3.0 at n = 10 in the simulation of §8.1. Planning a new study on that inflated effect guarantees it will be underpowered too.</details>

<details><summary>9. How can a completely null dataset yield a significant result a fifth of the time?</summary>If the analysis is chosen after seeing the data. Ten defensible analytic choices — a different test, an outlier rule, a subgroup, a transformation, stopping early — each with a 5 % false-positive rate and positively correlated with each other, give a 22.4 % chance that at least one is significant. Pre-specify the analysis, or correct over all the paths that were available.</details>

<details><summary>10. You have 3,000 MD frames and six runs. How many independent observations do you have?</summary>Six, for any question about the difference between conditions. The frames within a run are autocorrelated (Unit 7 measured integrated autocorrelation times between about 18 and 80 ps), so they contribute a few dozen effectively independent samples each; but the quantity that varies between repetitions of the experiment is the run, so the run is the unit.</details>

<details><summary>11. The chi-square test and Fisher's exact test disagree on your 2 × 2 table. Which do you trust?</summary>Fisher's, and the disagreement itself tells you why: the chi-square approximation degrades when expected cell counts fall below about 5. Check the expected counts (they are in <code>chisq.test(tbl)$expected</code>); if any is small, report the exact test and the odds ratio with its interval. In §3.2 the smallest expected count was 56, so the two agreed.</details>

<details><summary>12. Write the one-sentence result of the temperature comparison in this unit.</summary>"Across three independent 1 ns replicas per condition, the mean C-alpha RMSD was 0.174 ± 0.042 nm at 300 K and 0.153 ± 0.020 nm at 340 K (Welch t-test on replica means, t = 0.44, p = 0.69; exact permutation p = 0.80); with this between-replica variability the design had 4 % power for the observed 0.02 nm difference, so no structural difference was detectable at this timescale."</details>

## Glossary

<dl class="gloss">
<dt>Benjamini–Hochberg (BH)</dt><dd>Step-up procedure controlling the false discovery rate: reject the k smallest p-values where k is the largest rank with p<sub>(k)</sub> ≤ kα/m. <code>p.adjust(p, "BH")</code>.</dd>
<dt>Bonferroni correction</dt><dd>Test each of m hypotheses at α/m; controls the family-wise error rate, conservatively.</dd>
<dt>Cliff's delta</dt><dd>Non-parametric effect size: P(X &gt; Y) − P(X &lt; Y), ranging from −1 to 1; algebraically the rank-biserial correlation.</dd>
<dt>Cohen's d</dt><dd>Difference in means divided by the pooled standard deviation; Hedges' g is its small-sample correction.</dd>
<dt>Effect size</dt><dd>A measure of how large a difference is, in units comparable across studies; always reported with an interval.</dd>
<dt>Exchangeability</dt><dd>The assumption that, under the null, the labels can be permuted without changing the joint distribution; it defines which permutation scheme is valid (label shuffle for independent groups, sign flip for pairs).</dd>
<dt>False discovery rate (FDR)</dt><dd>Expected proportion of false positives among rejected hypotheses.</dd>
<dt>Family-wise error rate (FWER)</dt><dd>Probability of at least one false positive in a family of tests.</dd>
<dt>Garden of forking paths</dt><dd>Inflation of the false-positive rate caused by data-dependent analysis choices, even without any deliberate searching.</dd>
<dt>HARKing</dt><dd>Hypothesising after the results are known, then presenting the hypothesis as pre-specified.</dd>
<dt>Holm procedure</dt><dd>Step-down FWER correction: uniformly more powerful than Bonferroni with the same guarantee.</dd>
<dt>Null hypothesis</dt><dd>A complete, simulatable description of "no effect", against which the data are compared.</dd>
<dt>Odds ratio</dt><dd>Effect size for a 2 × 2 table: the ratio of the odds of an outcome in the two groups; its interval is built on the log scale.</dd>
<dt>Paired design</dt><dd>Each observation in one group has a natural partner in the other; analysis works on within-pair differences.</dd>
<dt>Permutation test</dt><dd>Null distribution built by relabelling the data; the general case of which the classical tests are shortcuts.</dd>
<dt>Power</dt><dd>Probability of rejecting the null when a specified effect is real; determined by effect size, variability, n and α. <code>power.t.test()</code> or simulation.</dd>
<dt>p-hacking</dt><dd>Searching analyses or stopping rules until a significant result appears.</dd>
<dt>Pseudoreplication</dt><dd>Treating repeated measurements of the same experimental unit as independent replicates.</dd>
<dt>q-value</dt><dd>The adjusted p-value of a BH-type procedure; the smallest FDR at which that hypothesis is rejected.</dd>
<dt>Rank-biserial correlation</dt><dd>Effect size derived from the Mann–Whitney statistic, 2U/(n₁n₂) − 1.</dd>
<dt>Welch's t-test</dt><dd>Two-sample t-test that does not assume equal variances; the sensible default and R's.</dd>
<dt>Winner's curse</dt><dd>Upward bias of effect estimates among results selected for significance in underpowered studies.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r2">Papapanou PN, Behle JH, Kebschull M, et al. Subgingival bacterial colonization profiles correlate with gingival tissue gene expression. <em>BMC Microbiol</em>. 2009;9:221. <a href="https://doi.org/10.1186/1471-2180-9-221">doi:10.1186/1471-2180-9-221</a></li>
<li id="r3">Ernst MD. Permutation methods: a basis for exact inference. <em>Stat Sci</em>. 2004;19(4):676-685. <a href="https://doi.org/10.1214/088342304000000396">doi:10.1214/088342304000000396</a></li>
<li id="r4">Welch BL. The generalization of "Student's" problem when several different population variances are involved. <em>Biometrika</em>. 1947;34(1-2):28-35. <a href="https://doi.org/10.1093/biomet/34.1-2.28">doi:10.1093/biomet/34.1-2.28</a></li>
<li id="r5">Mann HB, Whitney DR. On a test of whether one of two random variables is stochastically larger than the other. <em>Ann Math Stat</em>. 1947;18(1):50-60. <a href="https://doi.org/10.1214/aoms/1177730491">doi:10.1214/aoms/1177730491</a></li>
<li id="r6">Wilcoxon F. Individual comparisons by ranking methods. In: Kotz S, Johnson NL, eds. <em>Breakthroughs in Statistics</em>. Springer; 1992:196-202 (orig. <em>Biometrics Bull</em>. 1945;1(6):80-83). <a href="https://doi.org/10.1007/978-1-4612-4380-9_16">doi:10.1007/978-1-4612-4380-9_16</a></li>
<li id="r7">Kruskal WH, Wallis WA. Use of ranks in one-criterion variance analysis. <em>J Am Stat Assoc</em>. 1952;47(260):583-621. <a href="https://doi.org/10.2307/2280779">doi:10.2307/2280779</a></li>
<li id="r8">Fisher RA. On the interpretation of chi-square from contingency tables, and the calculation of P. <em>J R Stat Soc</em>. 1922;85(1):87-94. <a href="https://doi.org/10.2307/2340521">doi:10.2307/2340521</a></li>
<li id="r9">Wasserstein RL, Lazar NA. The ASA statement on p-values: context, process, and purpose. <em>Am Stat</em>. 2016;70(2):129-133. <a href="https://doi.org/10.1080/00031305.2016.1154108">doi:10.1080/00031305.2016.1154108</a></li>
<li id="r10">Greenland S, Senn SJ, Rothman KJ, et al. Statistical tests, P values, confidence intervals, and power: a guide to misinterpretations. <em>Eur J Epidemiol</em>. 2016;31:337-350. <a href="https://doi.org/10.1007/s10654-016-0149-3">doi:10.1007/s10654-016-0149-3</a></li>
<li id="r11">Halsey LG, Curran-Everett D, Vowler SL, Drummond GB. The fickle P value generates irreproducible results. <em>Nat Methods</em>. 2015;12:179-185. <a href="https://doi.org/10.1038/nmeth.3288">doi:10.1038/nmeth.3288</a></li>
<li id="r12">Amrhein V, Greenland S, McShane B. Scientists rise up against statistical significance. <em>Nature</em>. 2019;567:305-307. <a href="https://doi.org/10.1038/d41586-019-00857-9">doi:10.1038/d41586-019-00857-9</a></li>
<li id="r13">Nuzzo R. Scientific method: statistical errors. <em>Nature</em>. 2014;506:150-152. <a href="https://doi.org/10.1038/506150a">doi:10.1038/506150a</a></li>
<li id="r14">Lakens D. Calculating and reporting effect sizes to facilitate cumulative science: a practical primer for t-tests and ANOVAs. <em>Front Psychol</em>. 2013;4:863. <a href="https://doi.org/10.3389/fpsyg.2013.00863">doi:10.3389/fpsyg.2013.00863</a></li>
<li id="r15">Hedges LV. Distribution theory for Glass's estimator of effect size and related estimators. <em>J Educ Stat</em>. 1981;6(2):107-128. <a href="https://doi.org/10.3102/10769986006002107">doi:10.3102/10769986006002107</a></li>
<li id="r16">Vargha A, Delaney HD. A critique and improvement of the CL common language effect size statistics of McGraw and Wong. <em>J Educ Behav Stat</em>. 2000;25(2):101-132. <a href="https://doi.org/10.3102/10769986025002101">doi:10.3102/10769986025002101</a></li>
<li id="r17">Aickin M, Gensler H. Adjusting for multiple testing when reporting research results: the Bonferroni vs Holm methods. <em>Am J Public Health</em>. 1996;86(5):726-728. <a href="https://doi.org/10.2105/AJPH.86.5.726">doi:10.2105/AJPH.86.5.726</a></li>
<li id="r18">Benjamini Y, Hochberg Y. Controlling the false discovery rate: a practical and powerful approach to multiple testing. <em>J R Stat Soc B</em>. 1995;57(1):289-300. <a href="https://doi.org/10.1111/j.2517-6161.1995.tb02031.x">doi:10.1111/j.2517-6161.1995.tb02031.x</a></li>
<li id="r19">Storey JD, Tibshirani R. Statistical significance for genomewide studies. <em>Proc Natl Acad Sci USA</em>. 2003;100(16):9440-9445. <a href="https://doi.org/10.1073/pnas.1530509100">doi:10.1073/pnas.1530509100</a></li>
<li id="r20">Simmons JP, Nelson LD, Simonsohn U. False-positive psychology: undisclosed flexibility in data collection and analysis allows presenting anything as significant. <em>Psychol Sci</em>. 2011;22(11):1359-1366. <a href="https://doi.org/10.1177/0956797611417632">doi:10.1177/0956797611417632</a></li>
<li id="r21">Head ML, Holman L, Lanfear R, Kahn AT, Jennions MD. The extent and consequences of p-hacking in science. <em>PLoS Biol</em>. 2015;13(3):e1002106. <a href="https://doi.org/10.1371/journal.pbio.1002106">doi:10.1371/journal.pbio.1002106</a></li>
<li id="r22">Gelman A, Loken E. The statistical crisis in science. <em>Am Sci</em>. 2014;102(6):460-465. <a href="https://doi.org/10.1511/2014.111.460">doi:10.1511/2014.111.460</a></li>
<li id="r23">Kerr NL. HARKing: hypothesizing after the results are known. <em>Pers Soc Psychol Rev</em>. 1998;2(3):196-217. <a href="https://doi.org/10.1207/s15327957pspr0203_4">doi:10.1207/s15327957pspr0203_4</a></li>
<li id="r24">Nosek BA, Ebersole CR, DeHaven AC, Mellor DT. The preregistration revolution. <em>Proc Natl Acad Sci USA</em>. 2018;115(11):2600-2606. <a href="https://doi.org/10.1073/pnas.1708274114">doi:10.1073/pnas.1708274114</a></li>
<li id="r25">Munafò MR, Nosek BA, Bishop DVM, et al. A manifesto for reproducible science. <em>Nat Hum Behav</em>. 2017;1:0021. <a href="https://doi.org/10.1038/s41562-016-0021">doi:10.1038/s41562-016-0021</a></li>
<li id="r26">Button KS, Ioannidis JPA, Mokrysz C, et al. Power failure: why small sample size undermines the reliability of neuroscience. <em>Nat Rev Neurosci</em>. 2013;14:365-376. <a href="https://doi.org/10.1038/nrn3475">doi:10.1038/nrn3475</a></li>
<li id="r27">Ioannidis JPA. Why most published research findings are false. <em>PLoS Med</em>. 2005;2(8):e124. <a href="https://doi.org/10.1371/journal.pmed.0020124">doi:10.1371/journal.pmed.0020124</a></li>
<li id="r28">Hurlbert SH. Pseudoreplication and the design of ecological field experiments. <em>Ecol Monogr</em>. 1984;54(2):187-211. <a href="https://doi.org/10.2307/1942661">doi:10.2307/1942661</a></li>
<li id="r29">Lazic SE. The problem of pseudoreplication in neuroscientific studies: is it affecting your analysis? <em>BMC Neurosci</em>. 2010;11:5. <a href="https://doi.org/10.1186/1471-2202-11-5">doi:10.1186/1471-2202-11-5</a></li>
<li id="r30">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r31">Grossfield A, Zuckerman DM. Quantifying uncertainty and sampling quality in biomolecular simulations. <em>Annu Rep Comput Chem</em>. 2009;5:23-48. <a href="https://doi.org/10.1016/S1574-1400(09)00502-7">doi:10.1016/S1574-1400(09)00502-7</a></li>
<li id="r32">Colquhoun D. An investigation of the false discovery rate and the misinterpretation of p-values. <em>R Soc Open Sci</em>. 2014;1:140216. <a href="https://doi.org/10.1098/rsos.140216">doi:10.1098/rsos.140216</a></li>
<li id="r33">Krzywinski M, Altman N. Significance, P values and t-tests. <em>Nat Methods</em>. 2013;10:1041-1042. <a href="https://doi.org/10.1038/nmeth.2698">doi:10.1038/nmeth.2698</a></li>
<li id="r34">Krzywinski M, Altman N. Power and sample size. <em>Nat Methods</em>. 2013;10:1139-1140. <a href="https://doi.org/10.1038/nmeth.2738">doi:10.1038/nmeth.2738</a></li>
<li id="r35">Ben-Shachar MS, Lüdecke D, Makowski D. effectsize: estimation of effect size indices and standardized parameters. <em>J Open Source Softw</em>. 2020;5(56):2815. <a href="https://doi.org/10.21105/joss.02815">doi:10.21105/joss.02815</a></li>
<li id="r36">Fox J, Weisberg S. <em>An R Companion to Applied Regression</em>. 3rd ed. Sage; 2019. Software: the <code>car</code> package. <a href="https://doi.org/10.32614/CRAN.package.car">doi:10.32614/CRAN.package.car</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Wasserman L. <em>All of Statistics</em>. Springer; 2004. — Chapters 10–11 for testing at a mathematical level one step above this unit.</li>
<li>Good PI. <em>Permutation, Parametric and Bootstrap Tests of Hypotheses</em>. 3rd ed. Springer; 2005. — The book-length version of §2.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 (free PDF). — Chapter 13 on multiple testing, with the same BH derivation.</li>
<li>Cumming G. <em>Understanding the New Statistics</em>. Routledge; 2012. — Estimation-first reporting, the style this course asks for.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>R documentation for <code>p.adjust</code>, <code>power.t.test</code> and <code>t.test</code>: <a href="https://stat.ethz.ch/R-manual/R-devel/library/stats/html/p.adjust.html">stat.ethz.ch/R-manual</a>.</li>
<li>Nature Methods "Points of Significance" collection: <a href="https://www.nature.com/collections/qghhqm">nature.com/collections/qghhqm</a>.</li>
<li>Open Science Framework pre-registration templates: <a href="https://osf.io/prereg/">osf.io/prereg</a>.</li>
<li>LiveCoMS, "Best Practices for Quantification of Uncertainty and Sampling Quality in Molecular Simulations": <a href="https://livecomsjournal.org/">livecomsjournal.org</a>.</li>
</ul>
